# Pedestrian Network Height Assignment

This notebook creates a 3D pedestrian network for New York City by assigning heights from nearby elevation points.

The conservative rule used here is simple:

1. For each pedestrian network vertex, look only for height points within a local radius.
2. Average the heights of those local points.
3. If a vertex has no nearby height point, mark it invalid.
4. Drop any network segment that has at least one invalid vertex.

This avoids inventing slopes from height points that are too far away to be trusted.


## 1. Settings

The input pedestrian network is in New York Long Island StatePlane meters (`EPSG:6538`).

The height points are in New York Long Island StatePlane feet (`EPSG:2263`), so their XY coordinates are multiplied by the US survey foot-to-meter conversion before we compare distances.

Change `RADIUS_M` if you want a stricter or looser local search. The current value is `50 m`.


In [ ]:
from __future__ import annotations

import math
import shutil
import struct
import time
from collections import defaultdict
from dataclasses import dataclass
from datetime import date
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Sequence, Tuple

import numpy as np

PROJECT_DIR = Path.cwd()

NETWORK_SHP = PROJECT_DIR / "nyc_ped_net" / "nyc_ped_net.shp"
HEIGHT_SHP = PROJECT_DIR / "height_pnts" / "height_pnts.shp"
OUTPUT_PREFIX = PROJECT_DIR / "outputs" / "nyc_ped_net_radius50m"
QGIS_DIR = PROJECT_DIR / "qgis"

HEIGHT_FIELD = "ELEVATION"
RADIUS_M = 50.0
MIN_POINTS_PER_VERTEX = 1
CELL_SIZE_M = 25.0

US_SURVEY_FOOT_TO_METER = 0.30480060960121924

SHP_NULL = 0
SHP_POLYLINE = 3
SHP_POINTZ = 11
SHP_POLYLINEZ = 13
M_NODATA = -1.0e39


## 2. Small DBF Helpers

Shapefiles store attributes in `.dbf` files. We only need two DBF operations:

- read the `ELEVATION` column from the height points;
- write a very small output table with one field, `Z_M`.


### DBF metadata

Read the DBF header so we know how many records exist and where each field lives inside each row.


In [ ]:
@dataclass(frozen=True)
class DbfField:
    name: str
    field_type: str
    length: int
    decimals: int = 0
    offset: int = 0


@dataclass(frozen=True)
class DbfInfo:
    record_count: int
    header_length: int
    record_length: int
    fields: Tuple[DbfField, ...]


def dbf_path_for(shp_path: Path) -> Path:
    return shp_path.with_suffix(".dbf")


def shx_path_for(shp_path: Path) -> Path:
    return shp_path.with_suffix(".shx")


def read_dbf_info(dbf_path: Path) -> DbfInfo:
    with dbf_path.open("rb") as f:
        header = f.read(32)
        record_count = struct.unpack("<I", header[4:8])[0]
        header_length = struct.unpack("<H", header[8:10])[0]
        record_length = struct.unpack("<H", header[10:12])[0]

        fields = []
        offset = 1
        while True:
            desc = f.read(32)
            if desc[0] == 0x0D:
                break

            name = desc[:11].split(b"\0", 1)[0].decode("ascii", "ignore").strip()
            field_type = chr(desc[11])
            length = desc[16]
            decimals = desc[17]
            fields.append(DbfField(name, field_type, length, decimals, offset))
            offset += length

    return DbfInfo(record_count, header_length, record_length, tuple(fields))


def find_dbf_field(info: DbfInfo, field_name: str) -> DbfField:
    for field in info.fields:
        if field.name.upper() == field_name.upper():
            return field
    available = ", ".join(field.name for field in info.fields)
    raise ValueError(f"Field {field_name!r} not found. Available fields: {available}")


### DBF reading

Read one numeric column from the DBF file. Here that column is `ELEVATION` from the height point layer.


In [ ]:
def parse_dbf_number(raw: bytes) -> Optional[float]:
    text = raw.decode("ascii", "ignore").strip()
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def read_dbf_numeric_column(dbf_path: Path, field_name: str) -> List[Optional[float]]:
    info = read_dbf_info(dbf_path)
    field = find_dbf_field(info, field_name)
    values = []

    with dbf_path.open("rb") as f:
        f.seek(info.header_length)
        for _ in range(info.record_count):
            record = f.read(info.record_length)
            if record[0:1] == b"*":
                values.append(None)
                continue
            raw = record[field.offset : field.offset + field.length]
            values.append(parse_dbf_number(raw))

    return values


### DBF writing

Write a minimal DBF table for the outputs. We only keep one field: `Z_M`.


In [ ]:
def make_field(name: str, field_type: str, length: int, decimals: int = 0) -> DbfField:
    if len(name) > 10:
        raise ValueError("DBF field names in shapefiles can be at most 10 characters.")
    return DbfField(name, field_type, length, decimals)


def write_dbf_header(f, fields: Sequence[DbfField], record_count: int, record_length: int) -> None:
    header_length = 32 + 32 * len(fields) + 1
    today = date.today()

    header = bytearray(32)
    header[0] = 3
    header[1] = today.year - 1900
    header[2] = today.month
    header[3] = today.day
    struct.pack_into("<IHH", header, 4, record_count, header_length, record_length)
    f.write(header)

    for field in fields:
        desc = bytearray(32)
        name_bytes = field.name.encode("ascii")[:10]
        desc[: len(name_bytes)] = name_bytes
        desc[11] = ord(field.field_type)
        desc[16] = field.length
        desc[17] = field.decimals
        f.write(desc)

    f.write(b"\r")


def format_dbf_number(value: float, length: int, decimals: int) -> bytes:
    text = f"{value:.{decimals}f}" if decimals else str(int(round(value)))
    if len(text) > length:
        text = f"{value:.6E}"
    return text[:length].rjust(length).encode("ascii")


## 3. Small Shapefile Geometry Helpers

These helpers read the input geometries and write the output geometries. The output network is written as `LineString Z`, meaning every network vertex carries an elevation value inside the geometry.


### Shapefile headers

A shapefile starts with a fixed header. These helpers create headers for the output `LineString Z` and `Point Z` layers.


In [ ]:
@dataclass(frozen=True)
class PolylineRecord:
    record_number: int
    parts: Tuple[int, ...]
    points: Tuple[Tuple[float, float], ...]


def shapefile_header(
    shape_type: int,
    file_length_words: int,
    xy_bounds: Tuple[float, float, float, float],
    z_bounds: Tuple[float, float],
    m_bounds: Tuple[float, float] = (M_NODATA, M_NODATA),
) -> bytes:
    xmin, ymin, xmax, ymax = xy_bounds
    zmin, zmax = z_bounds
    mmin, mmax = m_bounds

    return struct.pack(
        ">7i", 9994, 0, 0, 0, 0, 0, file_length_words
    ) + struct.pack(
        "<2i8d", 1000, shape_type, xmin, ymin, xmax, ymax, zmin, zmax, mmin, mmax
    )


### Geometry readers

Read only the geometry we need: 2D network lines and 3D height points.


In [ ]:
def iter_polyline_records(shp_path: Path) -> Iterator[PolylineRecord]:
    with shp_path.open("rb") as f:
        f.seek(100)
        while True:
            record_header = f.read(8)
            if not record_header:
                break

            record_number, content_words = struct.unpack(">2i", record_header)
            content = f.read(content_words * 2)
            shape_type = struct.unpack("<i", content[:4])[0]
            if shape_type == SHP_NULL:
                continue
            if shape_type != SHP_POLYLINE:
                raise ValueError(f"Expected 2D PolyLine input, found shape type {shape_type}")

            num_parts, num_points = struct.unpack("<2i", content[36:44])
            parts_start = 44
            points_start = parts_start + 4 * num_parts
            parts = struct.unpack(f"<{num_parts}i", content[parts_start:points_start])

            points = []
            for idx in range(num_points):
                start = points_start + idx * 16
                points.append(struct.unpack("<2d", content[start : start + 16]))

            yield PolylineRecord(record_number, tuple(parts), tuple(points))


def iter_pointz_records(shp_path: Path) -> Iterator[Tuple[float, float, float]]:
    with shp_path.open("rb") as f:
        f.seek(100)
        while True:
            record_header = f.read(8)
            if not record_header:
                break

            _record_number, content_words = struct.unpack(">2i", record_header)
            content = f.read(content_words * 2)
            shape_type = struct.unpack("<i", content[:4])[0]
            if shape_type == SHP_NULL:
                continue
            if shape_type != SHP_POINTZ:
                raise ValueError(f"Expected PointZ height samples, found shape type {shape_type}")

            yield struct.unpack("<3d", content[4:28])


### Geometry writers

Prepare output geometries and copy the CRS sidecar files so QGIS reads the result correctly.


In [ ]:
def clean_existing_shapefile(output_shp: Path) -> None:
    for suffix in (".shp", ".shx", ".dbf", ".prj", ".cpg", ".qmd"):
        path = output_shp.with_suffix(suffix)
        if path.exists():
            path.unlink()


def copy_sidecars(source_shp: Path, output_shp: Path) -> None:
    for suffix in (".prj", ".cpg"):
        source = source_shp.with_suffix(suffix)
        if source.exists():
            shutil.copyfile(source, output_shp.with_suffix(suffix))


def polylinez_content(
    record: PolylineRecord,
    z_m: np.ndarray,
    z_ft: np.ndarray,
    vertex_ids: Dict[Tuple[float, float], int],
) -> bytes:
    xs = [point[0] for point in record.points]
    ys = [point[1] for point in record.points]
    zs = [float(z_m[vertex_ids[point]]) for point in record.points]

    content = bytearray()
    content.extend(struct.pack("<i", SHP_POLYLINEZ))
    content.extend(struct.pack("<4d", min(xs), min(ys), max(xs), max(ys)))
    content.extend(struct.pack("<2i", len(record.parts), len(record.points)))
    content.extend(struct.pack(f"<{len(record.parts)}i", *record.parts))

    for x, y in record.points:
        content.extend(struct.pack("<2d", x, y))

    content.extend(struct.pack("<2d", min(zs), max(zs)))
    for z in zs:
        content.extend(struct.pack("<d", z))

    content.extend(struct.pack("<2d", M_NODATA, M_NODATA))
    for _ in zs:
        content.extend(struct.pack("<d", M_NODATA))

    return bytes(content)


## 4. Read the Two Input Datasets

This cell reads:

- the height points and their `ELEVATION` values;
- the unique vertices of the pedestrian network.

The height point XY coordinates are converted from feet to meters so they can be compared to the network coordinates.


In [ ]:
def read_height_samples(
    height_shp: Path,
    height_field: str,
    xy_scale: float = US_SURVEY_FOOT_TO_METER,
    z_scale_to_m: float = US_SURVEY_FOOT_TO_METER,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    elevation_values = read_dbf_numeric_column(dbf_path_for(height_shp), height_field)

    xs = []
    ys = []
    z_ft = []
    z_m = []

    for idx, (x_raw, y_raw, z_raw) in enumerate(iter_pointz_records(height_shp)):
        elevation_ft = elevation_values[idx] if elevation_values[idx] is not None else z_raw
        xs.append(x_raw * xy_scale)
        ys.append(y_raw * xy_scale)
        z_ft.append(elevation_ft)
        z_m.append(elevation_ft * z_scale_to_m)

    return (
        np.asarray(xs, dtype=np.float64),
        np.asarray(ys, dtype=np.float64),
        np.asarray(z_ft, dtype=np.float64),
        np.asarray(z_m, dtype=np.float64),
    )


def collect_unique_network_vertices(
    network_shp: Path,
) -> Tuple[List[Tuple[float, float]], Dict[Tuple[float, float], int], int, int]:
    vertex_ids = {}
    feature_count = 0
    vertex_occurrences = 0

    for record in iter_polyline_records(network_shp):
        feature_count += 1
        vertex_occurrences += len(record.points)
        for point in record.points:
            if point not in vertex_ids:
                vertex_ids[point] = len(vertex_ids)

    vertices = list(vertex_ids.keys())
    return vertices, vertex_ids, feature_count, vertex_occurrences


height_x, height_y, height_z_ft, height_z_m = read_height_samples(HEIGHT_SHP, HEIGHT_FIELD)
vertices, vertex_ids, feature_count, vertex_occurrences = collect_unique_network_vertices(NETWORK_SHP)

print(f"Height samples: {len(height_x):,}")
print(f"Network features: {feature_count:,}")
print(f"Network vertex occurrences: {vertex_occurrences:,}")
print(f"Unique network vertices: {len(vertices):,}")


## 5. Build a Local Radius Search

For speed, the height points are placed in a simple grid. For each network vertex, we only inspect grid cells that could contain height points inside the chosen radius.

No height point outside `RADIUS_M` is used.


### Radius averaging class

This class returns the simple average height of all elevation points inside the chosen radius.


In [ ]:
@dataclass(frozen=True)
class RadiusResult:
    z_ft: float
    z_m: float
    sample_count: int
    nearest_m: float
    farthest_m: float


class RadiusAverageInterpolator:
    def __init__(
        self,
        x: np.ndarray,
        y: np.ndarray,
        z_ft: np.ndarray,
        z_m: np.ndarray,
        radius_m: float,
        cell_size_m: float,
    ) -> None:
        self.x = x
        self.y = y
        self.z_ft = z_ft
        self.z_m = z_m
        self.radius_m = radius_m
        self.radius2 = radius_m * radius_m
        self.cell_size = cell_size_m
        self.ring = max(1, int(math.ceil(radius_m / cell_size_m)))
        self.min_x = float(np.min(x))
        self.min_y = float(np.min(y))
        self.grid = defaultdict(list)

        for idx, (px, py) in enumerate(zip(x, y)):
            self.grid[self.cell_for(px, py)].append(idx)

    def cell_for(self, x: float, y: float) -> Tuple[int, int]:
        return (
            int(math.floor((x - self.min_x) / self.cell_size)),
            int(math.floor((y - self.min_y) / self.cell_size)),
        )

    def query(self, x: float, y: float) -> RadiusResult:
        ix, iy = self.cell_for(x, y)
        candidates = []

        for gx in range(ix - self.ring, ix + self.ring + 1):
            for gy in range(iy - self.ring, iy + self.ring + 1):
                candidates.extend(self.grid.get((gx, gy), ()))

        if not candidates:
            return RadiusResult(math.nan, math.nan, 0, math.nan, math.nan)

        idx = np.asarray(candidates, dtype=np.int64)
        dx = self.x[idx] - x
        dy = self.y[idx] - y
        d2 = dx * dx + dy * dy
        within = d2 <= self.radius2

        if not np.any(within):
            return RadiusResult(math.nan, math.nan, 0, math.nan, math.nan)

        local_idx = idx[within]
        distances = np.sqrt(d2[within])
        return RadiusResult(
            z_ft=float(np.mean(self.z_ft[local_idx])),
            z_m=float(np.mean(self.z_m[local_idx])),
            sample_count=int(len(local_idx)),
            nearest_m=float(np.min(distances)),
            farthest_m=float(np.max(distances)),
        )


### Build the search object

Now we build the local search grid once, then reuse it for every network vertex.


In [ ]:
interpolator = RadiusAverageInterpolator(
    height_x,
    height_y,
    height_z_ft,
    height_z_m,
    radius_m=RADIUS_M,
    cell_size_m=CELL_SIZE_M,
)

print(f"Radius search ready: using only points within {RADIUS_M:g} m")


## 6. Assign Heights to Network Vertices

Each unique network vertex receives the average height of all height points within `RADIUS_M`.

If a vertex has fewer than `MIN_POINTS_PER_VERTEX` height samples, it is marked invalid.


In [ ]:
def interpolate_vertices_by_radius(
    vertices: Sequence[Tuple[float, float]],
    interpolator: RadiusAverageInterpolator,
    min_points: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    count = len(vertices)
    z_ft = np.full(count, np.nan, dtype=np.float64)
    z_m = np.full(count, np.nan, dtype=np.float64)
    sample_count = np.zeros(count, dtype=np.int32)
    nearest_m = np.full(count, np.nan, dtype=np.float64)
    farthest_m = np.full(count, np.nan, dtype=np.float64)
    valid = np.zeros(count, dtype=bool)

    start = time.time()
    for idx, (x, y) in enumerate(vertices):
        result = interpolator.query(x, y)
        sample_count[idx] = result.sample_count

        if result.sample_count >= min_points:
            z_ft[idx] = result.z_ft
            z_m[idx] = result.z_m
            nearest_m[idx] = result.nearest_m
            farthest_m[idx] = result.farthest_m
            valid[idx] = True

        if (idx + 1) % 100_000 == 0:
            print(f"Processed {idx + 1:,}/{count:,} vertices in {time.time() - start:.1f}s")

    return z_ft, z_m, sample_count, nearest_m, farthest_m, valid


z_ft, z_m, sample_count, nearest_m, farthest_m, valid_vertex = interpolate_vertices_by_radius(
    vertices,
    interpolator,
    MIN_POINTS_PER_VERTEX,
)

valid_count = int(np.sum(valid_vertex))
print(f"Valid vertices: {valid_count:,} / {len(vertices):,} ({valid_count / len(vertices) * 100:.2f}%)")
print(f"Invalid vertices: {len(vertices) - valid_count:,}")


## 7. Keep Only Locally Supported Network Segments

A network segment is kept only if every vertex in that segment has a valid local height estimate.

This is the key conservative step: if one end of a segment does not have nearby height data, the whole segment is removed from the evaluation.


In [ ]:
@dataclass(frozen=True)
class NetworkValidity:
    flags: Tuple[bool, ...]
    kept: int
    dropped: int
    xy_bounds: Tuple[float, float, float, float]
    z_bounds_m: Tuple[float, float]


def compute_network_validity(
    network_shp: Path,
    vertex_ids: Dict[Tuple[float, float], int],
    valid_vertices: np.ndarray,
    z_m: np.ndarray,
) -> NetworkValidity:
    flags = []
    kept = 0
    dropped = 0
    xmin = ymin = zmin = math.inf
    xmax = ymax = zmax = -math.inf

    for record in iter_polyline_records(network_shp):
        ids = [vertex_ids[point] for point in record.points]
        keep = bool(np.all(valid_vertices[ids]))
        flags.append(keep)

        if not keep:
            dropped += 1
            continue

        kept += 1
        xs = [point[0] for point in record.points]
        ys = [point[1] for point in record.points]
        zs = z_m[ids]

        xmin = min(xmin, min(xs))
        ymin = min(ymin, min(ys))
        xmax = max(xmax, max(xs))
        ymax = max(ymax, max(ys))
        zmin = min(zmin, float(np.min(zs)))
        zmax = max(zmax, float(np.max(zs)))

    if kept == 0:
        raise ValueError("No network segments were retained. Try a larger radius.")

    return NetworkValidity(tuple(flags), kept, dropped, (xmin, ymin, xmax, ymax), (zmin, zmax))


network_validity = compute_network_validity(NETWORK_SHP, vertex_ids, valid_vertex, z_m)

print(f"Retained segments: {network_validity.kept:,}")
print(f"Dropped segments: {network_validity.dropped:,}")


## 8. Write the 3D Network

This section writes the retained pedestrian network as a `LineString Z` shapefile. This file is an intermediate output used by the routing export section.

The default deliverable for QGIS is now the GeoPackage created later in the notebook, not a collection of separate QA layers.


### Write the 3D network

Write retained segments as `LineString Z`. Each line also gets one `Z_M` attribute for easy styling in QGIS.


In [ ]:
def z_field() -> DbfField:
    return make_field("Z_M", "N", 18, 6)


def write_radius_network(output_shp: Path) -> None:
    clean_existing_shapefile(output_shp)
    output_shp.parent.mkdir(parents=True, exist_ok=True)

    output_shx = shx_path_for(output_shp)
    output_dbf = dbf_path_for(output_shp)
    field = z_field()
    record_length = 1 + field.length
    offsets = []

    with output_shp.open("wb") as shp_out, output_shx.open("wb") as shx_out, output_dbf.open("wb") as dbf_out:
        shp_out.write(shapefile_header(SHP_POLYLINEZ, 50, network_validity.xy_bounds, network_validity.z_bounds_m))
        shx_out.write(shapefile_header(SHP_POLYLINEZ, 50, network_validity.xy_bounds, network_validity.z_bounds_m))
        write_dbf_header(dbf_out, (field,), network_validity.kept, record_length)

        output_record_number = 1
        for input_index, record in enumerate(iter_polyline_records(NETWORK_SHP)):
            if not network_validity.flags[input_index]:
                continue

            content = polylinez_content(record, z_m, z_ft, vertex_ids)
            content_words = len(content) // 2
            offsets.append((shp_out.tell() // 2, content_words))
            shp_out.write(struct.pack(">2i", output_record_number, content_words))
            shp_out.write(content)

            ids = [vertex_ids[point] for point in record.points]
            segment_mean_z = float(np.mean(z_m[ids]))
            dbf_out.write(b" " + format_dbf_number(segment_mean_z, field.length, field.decimals))

            output_record_number += 1

        dbf_out.write(b"\x1a")

        shp_length_words = shp_out.tell() // 2
        shx_length_words = (100 + len(offsets) * 8) // 2
        for offset, length in offsets:
            shx_out.write(struct.pack(">2i", offset, length))

        shp_out.seek(0)
        shp_out.write(shapefile_header(SHP_POLYLINEZ, shp_length_words, network_validity.xy_bounds, network_validity.z_bounds_m))
        shx_out.seek(0)
        shx_out.write(shapefile_header(SHP_POLYLINEZ, shx_length_words, network_validity.xy_bounds, network_validity.z_bounds_m))

    copy_sidecars(NETWORK_SHP, output_shp)


### Optional QA vertex writer

The function below can write valid network vertices as `Point Z` for deeper elevation-assignment debugging. It is intentionally not called by the default workflow, because the standard deliverable should stay focused on the simplified routing GeoPackage.


In [ ]:
def write_radius_vertices(output_shp: Path) -> int:
    clean_existing_shapefile(output_shp)
    output_shp.parent.mkdir(parents=True, exist_ok=True)

    valid_indices = np.flatnonzero(valid_vertex)
    field = z_field()
    record_length = 1 + field.length
    offsets = []

    xs = np.asarray([vertices[idx][0] for idx in valid_indices], dtype=np.float64)
    ys = np.asarray([vertices[idx][1] for idx in valid_indices], dtype=np.float64)
    zs = z_m[valid_indices]
    xy_bounds = (float(np.min(xs)), float(np.min(ys)), float(np.max(xs)), float(np.max(ys)))
    z_bounds = (float(np.min(zs)), float(np.max(zs)))

    output_shx = shx_path_for(output_shp)
    output_dbf = dbf_path_for(output_shp)

    with output_shp.open("wb") as shp_out, output_shx.open("wb") as shx_out, output_dbf.open("wb") as dbf_out:
        shp_out.write(shapefile_header(SHP_POINTZ, 50, xy_bounds, z_bounds))
        shx_out.write(shapefile_header(SHP_POINTZ, 50, xy_bounds, z_bounds))
        write_dbf_header(dbf_out, (field,), len(valid_indices), record_length)

        for output_record_number, vertex_index in enumerate(valid_indices, start=1):
            x, y = vertices[int(vertex_index)]
            z_value = float(z_m[vertex_index])
            content = struct.pack("<i3d", SHP_POINTZ, x, y, z_value)
            content_words = len(content) // 2
            offsets.append((shp_out.tell() // 2, content_words))
            shp_out.write(struct.pack(">2i", output_record_number, content_words))
            shp_out.write(content)
            dbf_out.write(b" " + format_dbf_number(z_value, field.length, field.decimals))

        dbf_out.write(b"\x1a")

        shp_length_words = shp_out.tell() // 2
        shx_length_words = (100 + len(offsets) * 8) // 2
        for offset, length in offsets:
            shx_out.write(struct.pack(">2i", offset, length))

        shp_out.seek(0)
        shp_out.write(shapefile_header(SHP_POINTZ, shp_length_words, xy_bounds, z_bounds))
        shx_out.seek(0)
        shx_out.write(shapefile_header(SHP_POINTZ, shx_length_words, xy_bounds, z_bounds))

    copy_sidecars(NETWORK_SHP, output_shp)
    return int(len(valid_indices))


## 9. Run the 3D Network Export

This cell writes the intermediate 3D network used by the routing export section.


In [ ]:
network_output = Path(f"{OUTPUT_PREFIX}_network_z.shp")
report_output = Path(f"{OUTPUT_PREFIX}_report.txt")
valid_vertex_count = int(np.sum(valid_vertex))

write_radius_network(network_output)

print(f"3D network output: {network_output}")
print(f"Valid vertices used for height assignment: {valid_vertex_count:,}")


## 10. Write a Short Report

The report records the radius choice, counts, and simple diagnostics. This is useful later when comparing different radii such as 25 m, 50 m, or 75 m.


### Summarize diagnostics

These numbers tell us how much local evidence supports the retained network.


In [ ]:
def summarize_valid_vertices() -> str:
    local_nearest = nearest_m[valid_vertex]
    local_farthest = farthest_m[valid_vertex]
    local_samples = sample_count[valid_vertex]
    q_near = np.quantile(local_nearest, [0.5, 0.9, 0.99, 1.0])
    q_far = np.quantile(local_farthest, [0.5, 0.9, 0.99, 1.0])

    return (
        f"Valid vertices: {int(np.sum(valid_vertex)):,} / {len(valid_vertex):,} "
        f"({np.mean(valid_vertex) * 100:.2f}%).\n"
        f"Invalid vertices: {int(len(valid_vertex) - np.sum(valid_vertex)):,}.\n"
        f"Samples per valid vertex: min={int(np.min(local_samples))}, "
        f"median={float(np.median(local_samples)):.0f}, max={int(np.max(local_samples))}.\n"
        f"Nearest local sample distance (m): median={q_near[0]:.2f}, "
        f"p90={q_near[1]:.2f}, p99={q_near[2]:.2f}, max={q_near[3]:.2f}.\n"
        f"Farthest included sample distance (m): median={q_far[0]:.2f}, "
        f"p90={q_far[1]:.2f}, p99={q_far[2]:.2f}, max={q_far[3]:.2f}."
    )


### Save the report

Write a plain-text report next to the output layers.


In [ ]:
report_text = f"""Radius-limited pedestrian network height assignment report
===========================================================

Method
------
For each pedestrian network vertex, average only height points within {RADIUS_M} meters.
Vertices with fewer than {MIN_POINTS_PER_VERTEX} local height point(s) are invalid.
Network segments are retained only when every vertex in the segment is valid.

Inputs
------
Network: {NETWORK_SHP}
Height points: {HEIGHT_SHP}
Height field: {HEIGHT_FIELD}

Counts
------
Original network features: {feature_count:,}
Original network vertex occurrences: {vertex_occurrences:,}
Unique network vertices: {len(vertices):,}
Retained network segments: {network_validity.kept:,}
Dropped network segments: {network_validity.dropped:,}
Valid vertices used for height assignment: {valid_vertex_count:,}

Diagnostics
-----------
{summarize_valid_vertices()}

Outputs
-------
Intermediate 3D network: {network_output}
Final routing GeoPackage and CSVs are written in the routing export section.
"""

report_output.write_text(report_text, encoding="utf-8")
print(report_text)


## 11. Optional Validation

If `pyogrio` is available, this cell checks how GIS software will read the intermediate 3D network output.


In [ ]:
try:
    import pyogrio

    info = pyogrio.read_info(network_output)
    print(network_output.name)
    print("  geometry:", info["geometry_type"])
    print("  features:", info["features"])
    print("  CRS:", info["crs"])
    print("  fields:", list(info["fields"]))
except Exception as exc:
    print("Optional pyogrio validation skipped:", exc)


## 12. Build a Routing-Ready Topographic Network

The previous sections create a 3D pedestrian network. This section turns that network into a table that can be used by both QGIS and a shortest-path algorithm.

We are making one important project assumption here: the source pedestrian network already represents the accessible walking network. That means we do not need to identify inaccessible streets, sidewalks, or crossings from scratch. Instead, we keep the same physical network and add topographic attributes and scenario-based routing costs.

The exported `routing_network` layer has one row per retained physical segment. Each row stores:

- the segment geometry;
- the two endpoint nodes, `node_a` and `node_b`;
- elevation and grade attributes from A to B and from B to A;
- distance-only, slope-aware, and accessibility-sensitive costs for both directions.

This keeps the QGIS layer easy to read: no duplicate overlapping line features are needed just to represent direction. When we run a routing algorithm, the notebook expands each physical segment into two in-memory directed edges.


### Routing Cost Profiles

The cost profiles below are intentionally transparent scenario models. They are not a claim that we have measured the true accessibility burden for every user.

The thresholds come from a simple accessibility framing:

- `5%` grade (`1:20`) is treated as the upper end of a low-penalty walking surface;
- `8.33%` grade (`1:12`) is treated as a stronger threshold inspired by ramp guidance;
- above `8.33%`, the penalty rises more sharply.

These reference points come from the U.S. Access Board ADA Chapter 4 guidance on accessible routes and ramps: https://www.access-board.gov/ada/chapter/ch04/

The data here only measures running slope along the network. It does not measure cross slope, curb ramps, surface quality, sidewalk width, landings, obstructions, or pavement condition. For that reason, `cost_accessible_*` should be understood as a slope-aware accessibility scenario, not a full ADA accessibility certification.

One practical data note: very short geometry steps can produce unrealistic local grade percentages if two vertices are almost on top of each other but have different assigned elevations. For local maximum-grade reporting, we ignore steps shorter than `MIN_GRADE_STEP_LENGTH_M`. The full horizontal length, elevation gain, and elevation loss are still counted.


In [ ]:
import csv
import heapq
import sqlite3
from datetime import datetime, timezone

from shapely import to_wkb
from shapely.geometry import LineString, Point

ROUTING_PREFIX = Path(f"{OUTPUT_PREFIX}_routing")
routing_nodes_output = Path(f"{ROUTING_PREFIX}_nodes.csv")
routing_network_output = Path(f"{ROUTING_PREFIX}_network.csv")
routing_geopackage_output = QGIS_DIR / "nyc_ped_net_radius50m_routing.gpkg"

MIN_GRADE_STEP_LENGTH_M = 1.0
LOW_PENALTY_GRADE = 0.05
STRONG_PENALTY_GRADE = 1.0 / 12.0

SLOPE_AWARE_PROFILE = {
    "low_penalty_grade": LOW_PENALTY_GRADE,
    "strong_penalty_grade": STRONG_PENALTY_GRADE,
    "uphill_moderate_weight": 8.0,
    "uphill_steep_weight": 30.0,
    "downhill_moderate_weight": 2.0,
    "downhill_steep_weight": 8.0,
}

ACCESSIBLE_PROFILE = {
    "low_penalty_grade": LOW_PENALTY_GRADE,
    "strong_penalty_grade": STRONG_PENALTY_GRADE,
    "uphill_moderate_weight": 20.0,
    "uphill_steep_weight": 80.0,
    "downhill_moderate_weight": 12.0,
    "downhill_steep_weight": 50.0,
}


### Read the 3D Network Back From Disk

The next reader works from the exported `LineStringZ` shapefile instead of relying on temporary variables from earlier notebook cells. That makes this section reproducible on its own once the 3D network has been generated.


In [ ]:
@dataclass(frozen=True)
class PolylineZRecord:
    record_number: int
    parts: Tuple[int, ...]
    points: Tuple[Tuple[float, float, float], ...]


def iter_polylinez_records(shp_path: Path) -> Iterator[PolylineZRecord]:
    with shp_path.open("rb") as f:
        f.seek(100)
        while True:
            record_header = f.read(8)
            if not record_header:
                break

            record_number, content_words = struct.unpack(">2i", record_header)
            content = f.read(content_words * 2)
            shape_type = struct.unpack("<i", content[:4])[0]
            if shape_type == SHP_NULL:
                continue
            if shape_type != SHP_POLYLINEZ:
                raise ValueError(f"Expected 3D PolyLineZ input, found shape type {shape_type}")

            num_parts, num_points = struct.unpack("<2i", content[36:44])
            parts_start = 44
            points_start = parts_start + 4 * num_parts
            z_range_start = points_start + 16 * num_points
            z_values_start = z_range_start + 16

            parts = struct.unpack(f"<{num_parts}i", content[parts_start:points_start])
            xy_values = [
                struct.unpack("<2d", content[points_start + idx * 16 : points_start + (idx + 1) * 16])
                for idx in range(num_points)
            ]
            z_values = struct.unpack(
                f"<{num_points}d",
                content[z_values_start : z_values_start + 8 * num_points],
            )

            points = tuple((x, y, z) for (x, y), z in zip(xy_values, z_values))
            yield PolylineZRecord(record_number, tuple(parts), points)



def iter_pointz_records_from_z_layer(shp_path: Path) -> Iterator[Tuple[float, float, float]]:
    with shp_path.open("rb") as f:
        f.seek(100)
        while True:
            record_header = f.read(8)
            if not record_header:
                break

            _record_number, content_words = struct.unpack(">2i", record_header)
            content = f.read(content_words * 2)
            shape_type = struct.unpack("<i", content[:4])[0]
            if shape_type == SHP_NULL:
                continue
            if shape_type != SHP_POINTZ:
                raise ValueError(f"Expected PointZ input, found shape type {shape_type}")

            yield struct.unpack("<3d", content[4:28])


### Compute Segment Metrics and Scenario Costs

For each physical segment, we compute metrics in both directions.

The physical height difference is symmetric: if A to B gains 10 meters, then B to A loses 10 meters. The routing cost does not have to be symmetric, because cost is a scenario model of effort or accessibility burden. Uphill and downhill movement can receive different penalties.


In [ ]:
def grade_penalty_component(
    grade: float,
    low_penalty_grade: float,
    strong_penalty_grade: float,
    moderate_weight: float,
    steep_weight: float,
) -> float:
    moderate_excess = min(max(0.0, grade - low_penalty_grade), strong_penalty_grade - low_penalty_grade)
    steep_excess = max(0.0, grade - strong_penalty_grade)
    return moderate_weight * moderate_excess + steep_weight * steep_excess


def route_cost_from_profile(length_m: float, gain_m: float, loss_m: float, profile: Dict[str, float]) -> float:
    if length_m <= 0:
        return 0.0

    uphill_grade = gain_m / length_m
    downhill_grade = loss_m / length_m

    multiplier = 1.0
    multiplier += grade_penalty_component(
        uphill_grade,
        profile["low_penalty_grade"],
        profile["strong_penalty_grade"],
        profile["uphill_moderate_weight"],
        profile["uphill_steep_weight"],
    )
    multiplier += grade_penalty_component(
        downhill_grade,
        profile["low_penalty_grade"],
        profile["strong_penalty_grade"],
        profile["downhill_moderate_weight"],
        profile["downhill_steep_weight"],
    )
    return length_m * multiplier


def directed_geometry_metrics(points: Sequence[Tuple[float, float, float]]) -> Dict[str, float]:
    length_m = 0.0
    length_3d_m = 0.0
    gain_m = 0.0
    loss_m = 0.0
    max_uphill_grade = 0.0
    max_downhill_grade = 0.0
    grade_step_count = 0
    short_step_count = 0

    for start, end in zip(points, points[1:]):
        x1, y1, z1 = start
        x2, y2, z2 = end
        horizontal_m = math.hypot(x2 - x1, y2 - y1)
        dz_m = z2 - z1
        if horizontal_m <= 0:
            continue

        length_m += horizontal_m
        length_3d_m += math.hypot(horizontal_m, dz_m)

        step_grade = dz_m / horizontal_m
        if dz_m > 0:
            gain_m += dz_m
        elif dz_m < 0:
            loss_m += -dz_m

        if horizontal_m >= MIN_GRADE_STEP_LENGTH_M:
            grade_step_count += 1
            if step_grade > 0:
                max_uphill_grade = max(max_uphill_grade, step_grade)
            elif step_grade < 0:
                max_downhill_grade = max(max_downhill_grade, -step_grade)
        else:
            short_step_count += 1

    z_start_m = points[0][2]
    z_end_m = points[-1][2]
    net_change_m = z_end_m - z_start_m
    avg_grade = net_change_m / length_m if length_m else 0.0

    return {
        "length_m": length_m,
        "length_3d_m": length_3d_m,
        "z_start_m": z_start_m,
        "z_end_m": z_end_m,
        "net_elevation_change_m": net_change_m,
        "elevation_gain_m": gain_m,
        "elevation_loss_m": loss_m,
        "avg_grade_pct": avg_grade * 100.0,
        "max_uphill_grade_pct": max_uphill_grade * 100.0,
        "max_downhill_grade_pct": max_downhill_grade * 100.0,
        "grade_step_count": grade_step_count,
        "short_step_count": short_step_count,
    }


def rounded_metric(value: float, digits: int = 6) -> float:
    return round(float(value), digits)


### Build Routing Nodes and Physical Segment Rows

The node table stores the graph vertices. The network table stores one row per physical segment, with A-to-B and B-to-A metrics side by side.

The field names use `a_to_b` and `b_to_a` instead of `forward` and `reverse` because A and B are simply the two endpoints stored on each row. A is the first coordinate of the retained 3D line, and B is the last coordinate. This is a storage convention, not a real-world movement restriction.


In [ ]:
def build_routing_tables(
    network_z_shp: Path,
) -> Tuple[List[Dict[str, object]], List[Dict[str, object]], List[LineString]]:
    node_lookup: Dict[Tuple[float, float], int] = {}
    node_rows: List[Dict[str, object]] = []
    network_rows: List[Dict[str, object]] = []
    network_geometries: List[LineString] = []

    def node_id_for(point: Tuple[float, float, float]) -> int:
        x, y, z = point
        key = (x, y)
        if key not in node_lookup:
            node_id = len(node_rows)
            node_lookup[key] = node_id
            node_rows.append({
                "node_id": node_id,
                "x": rounded_metric(x, 3),
                "y": rounded_metric(y, 3),
                "z_m": rounded_metric(z, 6),
            })
        return node_lookup[key]

    for record in iter_polylinez_records(network_z_shp):
        points_ab = record.points
        points_ba = tuple(reversed(record.points))
        metrics_ab = directed_geometry_metrics(points_ab)
        metrics_ba = directed_geometry_metrics(points_ba)

        node_a = node_id_for(points_ab[0])
        node_b = node_id_for(points_ab[-1])
        length_m = metrics_ab["length_m"]
        max_abs_grade_pct = max(
            metrics_ab["max_uphill_grade_pct"],
            metrics_ab["max_downhill_grade_pct"],
        )

        network_rows.append({
            "segment_id": len(network_rows),
            "network_record_number": record.record_number,
            "node_a": node_a,
            "node_b": node_b,
            "point_count": len(points_ab),
            "length_m": rounded_metric(length_m),
            "z_a_m": rounded_metric(metrics_ab["z_start_m"]),
            "z_b_m": rounded_metric(metrics_ab["z_end_m"]),
            "net_change_a_to_b_m": rounded_metric(metrics_ab["net_elevation_change_m"]),
            "avg_grade_a_to_b_pct": rounded_metric(metrics_ab["avg_grade_pct"]),
            "max_abs_grade_pct": rounded_metric(max_abs_grade_pct),
            "cost_distance": rounded_metric(length_m),
            "cost_slope_a_to_b": rounded_metric(route_cost_from_profile(
                length_m,
                metrics_ab["elevation_gain_m"],
                metrics_ab["elevation_loss_m"],
                SLOPE_AWARE_PROFILE,
            )),
            "cost_slope_b_to_a": rounded_metric(route_cost_from_profile(
                length_m,
                metrics_ba["elevation_gain_m"],
                metrics_ba["elevation_loss_m"],
                SLOPE_AWARE_PROFILE,
            )),
            "cost_accessible_a_to_b": rounded_metric(route_cost_from_profile(
                length_m,
                metrics_ab["elevation_gain_m"],
                metrics_ab["elevation_loss_m"],
                ACCESSIBLE_PROFILE,
            )),
            "cost_accessible_b_to_a": rounded_metric(route_cost_from_profile(
                length_m,
                metrics_ba["elevation_gain_m"],
                metrics_ba["elevation_loss_m"],
                ACCESSIBLE_PROFILE,
            )),
        })
        network_geometries.append(LineString(points_ab))

    return node_rows, network_rows, network_geometries


def write_csv_rows(output_path: Path, rows: Sequence[Dict[str, object]]) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        raise ValueError(f"No rows to write for {output_path}")

    fieldnames = list(rows[0].keys())
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
routing_nodes, routing_network, routing_network_geometries = build_routing_tables(network_output)
write_csv_rows(routing_nodes_output, routing_nodes)
write_csv_rows(routing_network_output, routing_network)

segment_lengths = np.asarray([segment["length_m"] for segment in routing_network], dtype=np.float64)
max_abs_grades = np.asarray([segment["max_abs_grade_pct"] for segment in routing_network], dtype=np.float64)

print(f"Routing nodes: {len(routing_nodes):,} written to {routing_nodes_output}")
print(f"Physical routing network segments: {len(routing_network):,} written to {routing_network_output}")
print(f"Average segment length: {np.mean(segment_lengths):.2f} m")
print(f"Maximum local absolute grade after filtering short steps: {np.max(max_abs_grades):.2f}%")
print(f"99th percentile local absolute grade: {np.percentile(max_abs_grades, 99):.2f}%")


### Export a QGIS-Friendly GeoPackage

The CSVs are useful for routing algorithms, but QGIS needs spatial layers. The GeoPackage below is the GIS-friendly companion output.

The GeoPackage contains two layers:

- `routing_network`: one 3D line feature per physical pedestrian network segment, with A-to-B and B-to-A routing costs stored as attributes;
- `routing_nodes`: graph nodes referenced by `routing_network.node_a` and `routing_network.node_b`.

This means QGIS users can visualize physical streets once, without duplicate overlapping lines. The route solver can still expand each row into two directed graph edges in memory.

The writer uses the GeoPackage SQLite structure directly so this notebook does not depend on `geopandas` or `pandas` being available.


In [ ]:
def quote_sql_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def sqlite_type_for_value(value: object) -> str:
    if isinstance(value, (int, np.integer)):
        return "INTEGER"
    if isinstance(value, (float, np.floating)):
        return "REAL"
    return "TEXT"


def gpkg_geometry_blob(geometry, srs_id: int = 6538) -> sqlite3.Binary:
    # GeoPackageBinary header: magic, version, flags, SRS id, then standard little-endian WKB.
    header = b"GP" + bytes([0, 1]) + struct.pack("<i", srs_id)
    wkb = to_wkb(geometry, byte_order=1, output_dimension=3)
    return sqlite3.Binary(header + wkb)


def update_bounds(bounds: Optional[Tuple[float, float, float, float]], geometry) -> Tuple[float, float, float, float]:
    xmin, ymin, xmax, ymax = geometry.bounds
    if bounds is None:
        return (xmin, ymin, xmax, ymax)
    return (
        min(bounds[0], xmin),
        min(bounds[1], ymin),
        max(bounds[2], xmax),
        max(bounds[3], ymax),
    )


def initialize_geopackage(conn: sqlite3.Connection, srs_wkt: str) -> None:
    conn.execute("PRAGMA application_id = 1196444487")
    conn.execute("PRAGMA user_version = 10300")
    conn.execute("PRAGMA journal_mode = MEMORY")
    conn.execute("PRAGMA synchronous = OFF")

    conn.execute("""
        CREATE TABLE gpkg_spatial_ref_sys (
            srs_name TEXT NOT NULL,
            srs_id INTEGER NOT NULL PRIMARY KEY,
            organization TEXT NOT NULL,
            organization_coordsys_id INTEGER NOT NULL,
            definition TEXT NOT NULL,
            description TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE gpkg_contents (
            table_name TEXT NOT NULL PRIMARY KEY,
            data_type TEXT NOT NULL,
            identifier TEXT UNIQUE,
            description TEXT DEFAULT '',
            last_change DATETIME NOT NULL DEFAULT (strftime('%Y-%m-%dT%H:%M:%fZ', 'now')),
            min_x DOUBLE,
            min_y DOUBLE,
            max_x DOUBLE,
            max_y DOUBLE,
            srs_id INTEGER,
            CONSTRAINT fk_gc_r_srs_id FOREIGN KEY (srs_id) REFERENCES gpkg_spatial_ref_sys(srs_id)
        )
    """)
    conn.execute("""
        CREATE TABLE gpkg_geometry_columns (
            table_name TEXT NOT NULL,
            column_name TEXT NOT NULL,
            geometry_type_name TEXT NOT NULL,
            srs_id INTEGER NOT NULL,
            z TINYINT NOT NULL,
            m TINYINT NOT NULL,
            CONSTRAINT pk_geom_cols PRIMARY KEY (table_name, column_name),
            CONSTRAINT fk_gc_tn FOREIGN KEY (table_name) REFERENCES gpkg_contents(table_name),
            CONSTRAINT fk_gc_srs FOREIGN KEY (srs_id) REFERENCES gpkg_spatial_ref_sys(srs_id)
        )
    """)

    conn.executemany(
        """
        INSERT INTO gpkg_spatial_ref_sys
        (srs_name, srs_id, organization, organization_coordsys_id, definition, description)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        [
            ("Undefined Cartesian SRS", -1, "NONE", -1, "undefined", "undefined Cartesian coordinate reference system"),
            ("Undefined Geographic SRS", 0, "NONE", 0, "undefined", "undefined geographic coordinate reference system"),
            ("WGS 84 geodetic", 4326, "EPSG", 4326, "undefined", "longitude/latitude coordinates in decimal degrees"),
            ("NAD83(2011) / New York Long Island", 6538, "EPSG", 6538, srs_wkt, "New York Long Island StatePlane meters"),
        ],
    )


def write_feature_layer(
    conn: sqlite3.Connection,
    layer_name: str,
    geometry_type: str,
    field_schema: Sequence[Tuple[str, str]],
    row_geometry_iter: Iterator[Tuple[Dict[str, object], object]],
    srs_id: int = 6538,
    z: int = 1,
) -> int:
    table = quote_sql_identifier(layer_name)
    field_sql = ",\n            ".join(
        f"{quote_sql_identifier(field_name)} {field_type}" for field_name, field_type in field_schema
    )
    if field_sql:
        field_sql = ",\n            " + field_sql

    conn.execute(f"""
        CREATE TABLE {table} (
            fid INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
            geom {geometry_type} NOT NULL{field_sql}
        )
    """)
    conn.execute(
        "INSERT INTO gpkg_geometry_columns (table_name, column_name, geometry_type_name, srs_id, z, m) VALUES (?, ?, ?, ?, ?, ?)",
        (layer_name, "geom", geometry_type, srs_id, z, 0),
    )

    field_names = [field_name for field_name, _field_type in field_schema]
    insert_columns = ["geom"] + field_names
    placeholders = ", ".join("?" for _ in insert_columns)
    insert_sql = f"INSERT INTO {table} ({', '.join(quote_sql_identifier(col) for col in insert_columns)}) VALUES ({placeholders})"

    bounds = None
    count = 0
    for row, geometry in row_geometry_iter:
        values = [gpkg_geometry_blob(geometry, srs_id)] + [row[field_name] for field_name in field_names]
        conn.execute(insert_sql, values)
        bounds = update_bounds(bounds, geometry)
        count += 1
        if count % 100000 == 0:
            print(f"  {layer_name}: wrote {count:,} features")

    if bounds is None:
        raise ValueError(f"No features were written for layer {layer_name}")

    last_change = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")
    conn.execute(
        """
        INSERT INTO gpkg_contents
        (table_name, data_type, identifier, description, last_change, min_x, min_y, max_x, max_y, srs_id)
        VALUES (?, 'features', ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (layer_name, layer_name, layer_name, last_change, *bounds, srs_id),
    )
    return count


def routing_node_features(node_rows: Sequence[Dict[str, object]]) -> Iterator[Tuple[Dict[str, object], Point]]:
    for row in node_rows:
        yield row, Point(row["x"], row["y"], row["z_m"])


def write_routing_geopackage(
    output_path: Path,
    node_rows: Sequence[Dict[str, object]],
    network_rows: Sequence[Dict[str, object]],
    network_geometries: Sequence[LineString],
    network_z_shp: Path,
) -> Dict[str, int]:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        output_path.unlink()

    srs_wkt = network_z_shp.with_suffix(".prj").read_text(encoding="utf-8")
    counts = {}

    with sqlite3.connect(output_path) as conn:
        initialize_geopackage(conn, srs_wkt)

        network_schema = [(name, sqlite_type_for_value(network_rows[0][name])) for name in network_rows[0].keys()]
        node_schema = [(name, sqlite_type_for_value(node_rows[0][name])) for name in node_rows[0].keys()]

        counts["routing_network"] = write_feature_layer(
            conn,
            "routing_network",
            "LINESTRING",
            network_schema,
            zip(network_rows, network_geometries),
        )
        counts["routing_nodes"] = write_feature_layer(
            conn,
            "routing_nodes",
            "POINT",
            node_schema,
            routing_node_features(node_rows),
        )

    return counts


In [ ]:
geopackage_counts = write_routing_geopackage(
    routing_geopackage_output,
    routing_nodes,
    routing_network,
    routing_network_geometries,
    network_output,
)

print(f"GeoPackage written to {routing_geopackage_output}")
for layer_name, count in geopackage_counts.items():
    print(f"  {layer_name}: {count:,} features")


## 13. Prototype Route Comparison in the Notebook

The web application will eventually let users click two points on a map. Before building the app, we can test the routing idea directly in the notebook.

The helper functions below do three things:

1. snap an input coordinate to the nearest graph node;
2. expand the physical segment table into directed in-memory edges;
3. run Dijkstra shortest path using any selected cost profile.

This keeps the exported QGIS layer simple while preserving the direction-specific costs needed for routing.


In [ ]:
def directed_edges_from_network_rows(
    network_rows: Sequence[Dict[str, object]],
    cost_a_to_b_field: str,
    cost_b_to_a_field: str,
) -> List[Dict[str, object]]:
    directed_edges = []
    for row in network_rows:
        net_change = float(row["net_change_a_to_b_m"])
        gain_a_to_b = max(0.0, net_change)
        loss_a_to_b = max(0.0, -net_change)
        gain_b_to_a = loss_a_to_b
        loss_b_to_a = gain_a_to_b

        directed_edges.append({
            "directed_edge_id": len(directed_edges),
            "segment_id": int(row["segment_id"]),
            "network_record_number": int(row["network_record_number"]),
            "direction": "a_to_b",
            "source": int(row["node_a"]),
            "target": int(row["node_b"]),
            "cost": float(row[cost_a_to_b_field]),
            "length_m": float(row["length_m"]),
            "gain_m": gain_a_to_b,
            "loss_m": loss_a_to_b,
            "max_abs_grade_pct": float(row["max_abs_grade_pct"]),
        })
        directed_edges.append({
            "directed_edge_id": len(directed_edges),
            "segment_id": int(row["segment_id"]),
            "network_record_number": int(row["network_record_number"]),
            "direction": "b_to_a",
            "source": int(row["node_b"]),
            "target": int(row["node_a"]),
            "cost": float(row[cost_b_to_a_field]),
            "length_m": float(row["length_m"]),
            "gain_m": gain_b_to_a,
            "loss_m": loss_b_to_a,
            "max_abs_grade_pct": float(row["max_abs_grade_pct"]),
        })
    return directed_edges


def build_adjacency(directed_edges: Sequence[Dict[str, object]]) -> Dict[int, List[Tuple[int, float, int]]]:
    adjacency: Dict[int, List[Tuple[int, float, int]]] = defaultdict(list)
    for edge in directed_edges:
        adjacency[int(edge["source"])].append((
            int(edge["target"]),
            float(edge["cost"]),
            int(edge["directed_edge_id"]),
        ))
    return adjacency


node_xy = np.asarray([(node["x"], node["y"]) for node in routing_nodes], dtype=np.float64)


def nearest_node_id(x: float, y: float) -> int:
    distances2 = np.sum((node_xy - np.asarray([x, y])) ** 2, axis=1)
    return int(np.argmin(distances2))


def shortest_path_edges(
    adjacency: Dict[int, List[Tuple[int, float, int]]],
    start_node: int,
    end_node: int,
) -> Tuple[float, List[int]]:
    queue = [(0.0, start_node)]
    best_cost = {start_node: 0.0}
    previous: Dict[int, Tuple[int, int]] = {}

    while queue:
        current_cost, current_node = heapq.heappop(queue)
        if current_node == end_node:
            break
        if current_cost > best_cost.get(current_node, math.inf):
            continue

        for next_node, edge_cost, directed_edge_id in adjacency.get(current_node, []):
            proposed_cost = current_cost + edge_cost
            if proposed_cost < best_cost.get(next_node, math.inf):
                best_cost[next_node] = proposed_cost
                previous[next_node] = (current_node, directed_edge_id)
                heapq.heappush(queue, (proposed_cost, next_node))

    if end_node not in best_cost:
        return math.inf, []

    path_directed_edge_ids = []
    node = end_node
    while node != start_node:
        node, directed_edge_id = previous[node]
        path_directed_edge_ids.append(directed_edge_id)
    path_directed_edge_ids.reverse()

    return best_cost[end_node], path_directed_edge_ids


def summarize_route(directed_edges: Sequence[Dict[str, object]], directed_edge_ids: Sequence[int]) -> Dict[str, float]:
    edge_by_id = {int(edge["directed_edge_id"]): edge for edge in directed_edges}
    edges = [edge_by_id[edge_id] for edge_id in directed_edge_ids]
    if not edges:
        return {
            "edge_count": 0,
            "length_m": math.inf,
            "net_elevation_gain_m": math.inf,
            "net_elevation_loss_m": math.inf,
            "max_abs_grade_pct": math.inf,
        }

    return {
        "edge_count": len(edges),
        "length_m": sum(float(edge["length_m"]) for edge in edges),
        "net_elevation_gain_m": sum(float(edge["gain_m"]) for edge in edges),
        "net_elevation_loss_m": sum(float(edge["loss_m"]) for edge in edges),
        "max_abs_grade_pct": max(float(edge["max_abs_grade_pct"]) for edge in edges),
    }


def compare_routes(start_xy: Tuple[float, float], end_xy: Tuple[float, float]) -> Dict[str, Dict[str, float]]:
    start_node = nearest_node_id(*start_xy)
    end_node = nearest_node_id(*end_xy)

    comparisons = {}
    for label, cost_a_to_b_field, cost_b_to_a_field in [
        ("distance", "cost_distance", "cost_distance"),
        ("slope_aware", "cost_slope_a_to_b", "cost_slope_b_to_a"),
        ("accessible", "cost_accessible_a_to_b", "cost_accessible_b_to_a"),
    ]:
        directed_edges = directed_edges_from_network_rows(routing_network, cost_a_to_b_field, cost_b_to_a_field)
        adjacency = build_adjacency(directed_edges)
        total_cost, path_directed_edge_ids = shortest_path_edges(adjacency, start_node, end_node)
        summary = summarize_route(directed_edges, path_directed_edge_ids)
        summary["routing_cost"] = total_cost
        summary["start_node"] = start_node
        summary["end_node"] = end_node
        comparisons[label] = summary

    return comparisons


### Try One Origin-Destination Pair

The coordinates below are placeholders in the projected coordinate system used by the network. Replace them with any two map coordinates from the same CRS to test a specific route.

For the web application, this same pattern will be used after the user clicks two map locations: snap each click to the nearest graph node, run the three cost profiles, and return the route geometries plus these summary statistics.


In [ ]:
# Example: use two existing graph nodes so this cell works immediately after the routing table is built.
# Replace these with chosen origin/destination coordinates when testing specific neighborhoods.
example_start_xy = tuple(node_xy[0])
example_end_xy = tuple(node_xy[min(1000, len(node_xy) - 1)])

route_comparison = compare_routes(example_start_xy, example_end_xy)
for route_name, summary in route_comparison.items():
    print(route_name)
    for key, value in summary.items():
        if isinstance(value, float):
            print(f"  {key}: {value:,.2f}")
        else:
            print(f"  {key}: {value}")
